# 8교시. 실무 적용 시나리오 설계 및 최종 정리

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/leecks1119/document_ai_lecture/blob/master/colab/08_business_application.ipynb)

## 오늘 꼭 할 일

영수증이 아닌 업무 문서 사진 한 장에도 실제 VLM을 실행하고 작은 PoC를 정합니다.

1. 제공 예제로 결과를 먼저 만듭니다.
2. 화면에서 이번 교시의 핵심 결과 한 가지를 확인합니다.
3. 시간이 남으면 다른 공개·비식별 자료로 반복하고 차이를 기록합니다.

**끝났다는 증거:** 화면의 `✅ 실습 완료`와
`course_outputs/poc_candidate_card.md` 파일

> Google Colab도 외부 클라우드입니다. 조직 승인 없는 개인·회사 문서는
> 업로드하지 않습니다. 필수 실습은 저장소의 비식별 공개·합성 샘플만
> 사용합니다.

필수 실습에서는 제공 예제를 사용합니다. 다른 자료를 사용한 경우에는
화면에 표시된 파일명이 내가 선택한 파일과 같은지 먼저 확인합니다.
이 교시는 제공 이미지를 실제 VLM으로 읽어야 완료됩니다. GPU가 없거나
모델 실행이 실패하면 성공 결과로 바꾸지 않습니다. T4 GPU를
선택하고 모델 셀부터 다시 실행합니다.


## 이 노트북에서 내가 하는 일

- **필수 실습:** 견적서·신청서·거래명세서 사진 한 장을 실제 VLM으로 읽고 작은 PoC 카드를 만듭니다.
- **내가 바꾸는 곳:** 문서 종류·검토자·자동 저장 중단 조건만 정합니다.
- **인터넷 자료로 다시 실험:** 공개 이미지나 비식별 캡처 한 장으로 VLM을 다시 실행해 필요한 필드와 실패 조건을 기록합니다.

먼저 제공 예제로 끝까지 실행해 `✅ 실습 완료`를 확인하세요. 그다음
[공개·비식별 실습 자료 찾기](https://github.com/leecks1119/document_ai_lecture/blob/master/docs/public_practice_sources.md)를 보고
입력 한 장만 바꾸어 다시 실행합니다. 2교시에서 만든 결과 파일은
3~7교시에 이어 쓸 수 있습니다. 매 교시 마지막의 **다른 자료 실험
기록**에서 잘된 점과 실패한 점을 남깁니다.

> `🟢 그대로 실행하는 셀`은 수정하지 않습니다. `🟠 내가 짧게 바꾸는
> 셀`만 필수이고, `🔵 원하면 바꾸는 셀`은 시간이 남을 때 합니다.
> 정답은 모두 공개되어 있으므로 정답을 먼저 복사하고 결과를 관찰해도 됩니다.

## 코드 셀을 읽는 방법

각 코드 셀의 맨 위에는 `코드 읽기` 주석이 있습니다.

1. `수정하지 않습니다`라고 적힌 셀은 설명을 읽고 그대로 실행합니다.
2. 주황색 필수 `TODO`만 채웁니다. 파란색 선택 `TODO`는 건너뛰어도 됩니다.
3. 실행 출력에서 `코드 읽는 법`과 `확인할 결과`를 다시 확인합니다.
4. `단계 실행 완료`가 나온 뒤 다음 코드 셀로 이동합니다.
5. 길고 어려운 준비 코드는 접혀 있습니다. 제목 왼쪽의 화살표를 눌러
   펼칠 수 있지만, 처음에는 펼치지 않아도 됩니다.

Python 문법 전체를 먼저 이해할 필요는 없습니다. 변수에 어떤 값이 들어가고,
실행 뒤 어떤 결과가 달라지는지를 중심으로 읽습니다.


In [ ]:
#@title 🟢 0. 실습 환경 준비 — 그대로 실행 { display-mode: "form" }
def _show_learning_message(markdown_text):
    try:
        from IPython.display import Markdown, display
        display(Markdown(markdown_text))
    except ImportError:
        print(markdown_text)


def show_lab_step(
    current,
    total,
    title,
    action,
    expected,
    code_help,
    edit_kind,
):
    cell_kind = {
        "required": "🟠 내가 짧게 바꾸는 셀",
        "optional": "🔵 원하면 바꾸는 셀",
        "none": "🟢 그대로 실행하는 셀",
    }[edit_kind]
    _show_learning_message(
        f"""---
### {cell_kind} · {current}/{total} · {title}

**지금 할 일:** {action}

**코드 읽는 법:** {code_help}

**이 단계에서 확인할 결과:** {expected}
"""
    )


def complete_lab_step(current, total, expected):
    next_action = (
        "결과를 확인한 뒤 다음 코드 셀을 실행하세요."
        if current < total
        else "마지막 실습 완료 문구와 산출물 파일을 확인하세요."
    )
    _show_learning_message(
        f"""> ✅ **{current}/{total} 단계 실행 완료**
>
> **결과 확인:** {expected}
>
> **다음 행동:** {next_action}
"""
    )

# ── 코드 읽기 ─────────────────────────────────────────────
# `OUTPUT_DIR`와 Colab 공통 기능을 준비합니다. 수정하지 않습니다.
# ──────────────────────────────────────────────────────────
show_lab_step(1, 5, '공통 환경 준비', '업무 문서 결과를 저장할 환경을 준비합니다.', '공통 작업 폴더가 표시되어야 합니다.', '`OUTPUT_DIR`와 Colab 공통 기능을 준비합니다. 수정하지 않습니다.', 'none')

import json
import os
import platform
import sys
from pathlib import Path

OUTPUT_DIR = Path("course_outputs")
OUTPUT_DIR.mkdir(exist_ok=True)
AUTOMATED_CHECK = os.getenv("COURSE_VALIDATE_EXAMPLE") == "1"

def upload_previous_artifact(filename):
    target = OUTPUT_DIR / filename
    if target.exists() or AUTOMATED_CHECK:
        return target if target.exists() else None
    try:
        from google.colab import files
    except ImportError:
        return None
    print(f"이전 교시에서 내려받은 {filename}을 선택하세요.")
    uploaded = files.upload()
    if filename not in uploaded:
        raise FileNotFoundError(
            f"{filename}이 선택되지 않았습니다. 자료 선택에서 "
            "'제공 예제'를 고르거나 파일을 다시 선택하세요."
        )
    target.write_bytes(uploaded[filename])
    return target


def download_artifact(path):
    if AUTOMATED_CHECK:
        return
    try:
        from google.colab import files
    except ImportError:
        return
    files.download(str(path))

print("Python:", sys.version.split()[0])
print("Platform:", platform.platform())
print("공통 작업 폴더:", OUTPUT_DIR.resolve())

COURSE_ASSET_BASE_URL = (
    "https://raw.githubusercontent.com/leecks1119/"
    "document_ai_lecture/master/"
)

def load_course_assets(*relative_paths):
    if AUTOMATED_CHECK:
        local_root = os.getenv("COURSE_LOCAL_ASSET_ROOT")
        if not local_root:
            raise RuntimeError(
                "자동 검증용 COURSE_LOCAL_ASSET_ROOT가 필요합니다."
            )
        root = Path(local_root)
        return {
            path: (root / path).read_bytes()
            for path in relative_paths
        }

    import requests

    loaded = {}
    missing = []
    for path in relative_paths:
        try:
            response = requests.get(
                COURSE_ASSET_BASE_URL + path,
                timeout=30,
            )
            response.raise_for_status()
            loaded[path] = response.content
        except requests.RequestException as exc:
            print(f"자동 다운로드 실패: {Path(path).name} · {exc}")
            missing.append(path)

    if missing:
        from google.colab import files

        expected = ", ".join(Path(path).name for path in missing)
        print("다음 파일을 저장소에서 내려받아 선택하세요:", expected)
        uploaded = files.upload()
        uploaded_by_name = {
            Path(name).name: content
            for name, content in uploaded.items()
        }
        for path in missing:
            filename = Path(path).name
            if filename not in uploaded_by_name:
                raise FileNotFoundError(
                    f"{filename}이 선택되지 않았습니다."
                )
            loaded[path] = uploaded_by_name[filename]

    return loaded

complete_lab_step(1, 5, '공통 작업 폴더가 표시되어야 합니다.')


In [ ]:
#@title 🔵 실습 자료 고르기 { display-mode: "form" }
# ── 코드 읽기 ─────────────────────────────────────────────
# `문서_종류`와 `실습_자료`에서 실제 VLM 입력 한 장만 고릅니다.
# ──────────────────────────────────────────────────────────
show_lab_step(2, 5, '업무 문서 선택', '견적서·신청서·거래명세서 사진 한 장을 고릅니다.', '실제 VLM 입력과 Office 체험 묶음을 확인합니다.', '`문서_종류`와 `실습_자료`에서 실제 VLM 입력 한 장만 고릅니다.', 'optional')

COURSE_PYTHON_PATHS = ['src/__init__.py', 'src/clean.py', 'src/export.py', 'src/extract.py', 'src/ocr.py', 'src/pipeline.py', 'src/sample_data.py', 'src/validate.py', 'src/vlm.py']
course_python_assets = load_course_assets(*COURSE_PYTHON_PATHS)
for relative_path, payload in course_python_assets.items():
    target = Path(relative_path)
    target.parent.mkdir(parents=True, exist_ok=True)
    target.write_bytes(payload)
if str(Path.cwd()) not in sys.path:
    sys.path.insert(0, str(Path.cwd()))

# INPUT_FORM_CELL
import io
import requests
import zipfile
from PIL import Image
try:
    from IPython.display import display
except ImportError:
    display = print
from src.vlm import parse_with_paddleocr_vl, vlm_text_from_result

PHOTO_PATHS = {'견적서': 'sample_docs/extensions/quotation_photo.png', '신청서': 'sample_docs/extensions/application_form_photo.png', '거래명세서': 'sample_docs/extensions/transaction_statement_photo.png'}
OFFICE_PATHS = ['sample_docs/formats/quotation.xlsx', 'sample_docs/formats/application_form.docx', 'sample_docs/formats/transaction_statement.pdf', 'sample_docs/formats/table_summary.pptx']

# TODO(선택): 문서 종류 또는 입력 이미지를 바꾸어 다시 실행하세요.
문서_종류 = "거래명세서" #@param ["견적서", "신청서", "거래명세서"]
실습_자료 = "제공 예제" #@param ["제공 예제", "내 컴퓨터에서 업로드", "인터넷 이미지 URL"]
인터넷_이미지_URL = "" #@param {type:"string"}
if AUTOMATED_CHECK:
    실습_자료 = "제공 예제"

selected_path = PHOTO_PATHS[문서_종류]
input_bytes = load_course_assets(selected_path)[selected_path]
INPUT_FILE_NAME = Path(selected_path).name
if 실습_자료 == "내 컴퓨터에서 업로드":
    from google.colab import files
    uploaded = files.upload()
    if len(uploaded) != 1:
        raise ValueError("이미지 한 장만 선택하세요.")
    INPUT_FILE_NAME, input_bytes = next(iter(uploaded.items()))
elif 실습_자료 == "인터넷 이미지 URL":
    response = requests.get(인터넷_이미지_URL.strip(), timeout=30)
    response.raise_for_status()
    input_bytes = response.content
    INPUT_FILE_NAME = "internet_business_document.png"

input_image = Image.open(io.BytesIO(input_bytes)).convert("RGB")
INPUT_PATH = OUTPUT_DIR / "business_document.png"
input_image.save(INPUT_PATH)
preview = input_image.copy()
preview.thumbnail((650, 750))
display(preview)
print("실제 VLM 입력:", 문서_종류, INPUT_FILE_NAME)

office_assets = load_course_assets(*OFFICE_PATHS)
office_bundle = OUTPUT_DIR / "office_format_samples.zip"
with zipfile.ZipFile(office_bundle, "w") as archive:
    for path, payload in office_assets.items():
        target = OUTPUT_DIR / Path(path).name
        target.write_bytes(payload)
        archive.write(target, target.name)
print("Excel·Word·PDF·PPT 체험 묶음:", office_bundle)
download_artifact(office_bundle)

complete_lab_step(2, 5, '실제 VLM 입력과 Office 체험 묶음을 확인합니다.')


In [ ]:
#@title 🟢 업무 문서 VLM 실행 — 그대로 실행 { display-mode: "form" }
# ── 코드 읽기 ─────────────────────────────────────────────
# `parse_with_paddleocr_vl()`이 영수증이 아닌 업무 문서를 실제로 읽습니다.
# ──────────────────────────────────────────────────────────
show_lab_step(3, 5, '업무 문서 VLM 실행', 'T4 GPU에서 선택한 문서를 실제 VLM으로 읽습니다.', '실제 모델 여부와 Markdown 앞부분을 확인합니다.', '`parse_with_paddleocr_vl()`이 영수증이 아닌 업무 문서를 실제로 읽습니다.', 'none')

import subprocess

VLM_PIPELINE_NAME = "PaddleOCR-VL-1.6"
VLM_MODEL_NAME = "PaddleOCR-VL-1.6-0.9B"
if not AUTOMATED_CHECK:
    import torch
    if not torch.cuda.is_available():
        raise RuntimeError(
            "T4 GPU가 필요합니다. 런타임 → 런타임 유형 변경에서 "
            "T4 GPU를 선택한 뒤 이 셀부터 다시 실행하세요."
        )
    print("GPU:", torch.cuda.get_device_name(0))
    subprocess.check_call([
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "paddleocr[doc-parser]==3.7.0",
        "transformers>=5.8,<6",
    ])

if AUTOMATED_CHECK:
    business_vlm_result = {
        "model_executed": False,
        "pipeline": VLM_PIPELINE_NAME,
        "vlm_model": VLM_MODEL_NAME,
        "engine": "transformers",
        "input_file": INPUT_FILE_NAME,
        "pages": [],
    }
else:
    business_vlm_result = parse_with_paddleocr_vl(
        INPUT_PATH,
        engine="transformers",
        device="gpu",
    )

BUSINESS_MARKDOWN = vlm_text_from_result(business_vlm_result)
raw_path = OUTPUT_DIR / "business_vlm_raw.json"
raw_path.write_text(
    json.dumps(business_vlm_result, ensure_ascii=False, indent=2) + "\n",
    encoding="utf-8",
)
markdown_path = OUTPUT_DIR / "business_vlm_result.md"
if business_vlm_result["model_executed"]:
    markdown_path.write_text(BUSINESS_MARKDOWN + "\n", encoding="utf-8")

print("실제 모델 실행:", business_vlm_result["model_executed"])
print("모델:", VLM_MODEL_NAME)
print("\n--- 업무 문서에서 읽은 내용 앞부분 ---")
print(BUSINESS_MARKDOWN[:2200] or "자동검사에서는 거대 모델 실행만 생략합니다.")

complete_lab_step(3, 5, '실제 모델 여부와 Markdown 앞부분을 확인합니다.')


In [ ]:
# ── 코드 읽기 ─────────────────────────────────────────────
# `검토자`와 `중단_조건`만 입력해 모델 결과가 포함된 PoC 카드를 만듭니다.
# ──────────────────────────────────────────────────────────
show_lab_step(4, 5, 'PoC 카드 완성', '검토자·중단 조건을 정하고 작은 PoC 카드를 저장합니다.', '`poc_candidate_card.md`와 `✅ 실습 완료`를 확인합니다.', '`검토자`와 `중단_조건`만 입력해 모델 결과가 포함된 PoC 카드를 만듭니다.', 'required')

# TODO: 내 업무의 검토자와 자동 저장 중단 조건을 적으세요.
검토자 = "" #@param {type:"string"}
중단_조건 = "" #@param {type:"string"}

examples = {'quotation': {'name': '견적서', 'fields': ['문서번호', '공급자', '수신', '견적일', '품목', '총액'], 'rules': ['수량×단가=품목금액', '공급가액+부가세=총액'], 'risk': '총액 오류는 구매 의사결정에 직접 영향'}, 'application': {'name': '신청서', 'fields': ['신청번호', '신청자', '소속', '신청 과정', '승인'], 'rules': ['필수 동의', '관리자 승인 상태'], 'risk': '개인정보와 승인 누락을 사람이 확인'}, 'transaction_statement': {'name': '거래명세서', 'fields': ['문서번호', '공급자', '거래일', '품목', '세액', '총액'], 'rules': ['품목 합계=공급가액', '공급가액+세액=총액'], 'risk': '표 행·열 대응이 어긋나면 정산 오류'}}
key_by_name = {
    "견적서": "quotation",
    "신청서": "application",
    "거래명세서": "transaction_statement",
}
example = examples[key_by_name[문서_종류]]
검토자 = 검토자 or "업무 담당자"
중단_조건 = 중단_조건 or "필수값이나 원문 근거가 없으면 Excel 저장 중단"
print("전체 정답 예시:", 검토자, "/", 중단_조건)

card = f'''# 문서 자동화 PoC 후보 카드

| 항목 | 내용 |
| --- | --- |
| 대상 문서 | {example["name"]} |
| 이번 입력 | {INPUT_FILE_NAME} |
| 실제 VLM 실행 | {business_vlm_result["model_executed"]} |
| 사용할 모델 | {VLM_MODEL_NAME} |
| 추출 후보 | {", ".join(example["fields"])} |
| 검증 규칙 | {" / ".join(example["rules"])} |
| 사람 검토자 | {검토자} |
| 자동 저장 중단 | {중단_조건} |
| 최종 산출물 | 사람 확인 후 Excel |

## 이번 모델 결과에서 먼저 확인할 것

{BUSINESS_MARKDOWN[:800] or "Colab 실제 실행에서 모델 Markdown을 확인합니다."}
'''
output_path = OUTPUT_DIR / "poc_candidate_card.md"
output_path.write_text(card + "\n", encoding="utf-8")
print(card)
print("✅ 실습 완료:", output_path)
download_artifact(output_path)

complete_lab_step(4, 5, '`poc_candidate_card.md`와 `✅ 실습 완료`를 확인합니다.')


## 선택 실험: 다른 자료로 한 번 더 확인하기

필수 실습을 먼저 끝낸 뒤, 인터넷에서 찾은 공개 문서나 개인정보를
가린 자료 한 장으로 같은 과정을 반복합니다. 결과가 잘 나오지 않아도
실패한 위치와 다음 질문을 남기면 실험이 완료됩니다.


In [ ]:
#@title 🔵 선택: 다른 자료 실험 기록 { display-mode: "form" }
# ── 코드 읽기 ─────────────────────────────────────────────
# 이 셀은 선택 실험 기록지입니다. 위쪽 입력칸만 채우면 자료 출처, 잘된 점, 실패한 점, 다음 질문을 Markdown 파일로 저장합니다.
# ──────────────────────────────────────────────────────────
show_lab_step(5, 5, '다른 자료 실험 기록', '인터넷에서 찾은 공개 자료나 비식별 자료의 결과를 네 줄로 정리합니다.', '`lesson08_research_note.md` 파일과 기록 내용이 표시되어야 합니다.', '이 셀은 선택 실험 기록지입니다. 위쪽 입력칸만 채우면 자료 출처, 잘된 점, 실패한 점, 다음 질문을 Markdown 파일로 저장합니다.', 'optional')

# RESEARCH_NOTE_CELL
# TODO(선택): 다른 자료로 다시 실험했다면 아래 입력칸만 채우세요.
자료_구분 = "제공 예제" #@param ["제공 예제", "공개 웹 자료", "비식별 개인 자료", "회사 승인 자료"]
자료_이름_또는_URL = "" #@param {type:"string"}
문서_종류 = "영수증" #@param ["영수증", "견적서", "신청서", "거래명세서", "표 캡처", "기타"]
잘된_점 = "" #@param {type:"string"}
실패한_점 = "" #@param {type:"string"}
다음_질문 = "" #@param {type:"string"}

research_focus = '조사한 문서가 작은 PoC에 적합한 이유와 중단해야 할 조건을 기록합니다.'
note = f'''# {문서_종류} 실험 기록

- 자료 구분: {자료_구분}
- 자료 이름 또는 원문 URL: {자료_이름_또는_URL or "미입력"}
- 이번 교시 관찰 질문: {research_focus}
- 잘된 점: {잘된_점 or "미입력"}
- 실패하거나 이상한 점: {실패한_점 or "미입력"}
- 다음에 바꿔 볼 한 가지: {다음_질문 or "미입력"}
'''
note_path = OUTPUT_DIR / "lesson08_research_note.md"
note_path.write_text(note + "\n", encoding="utf-8")
try:
    from IPython.display import Markdown, display
    display(Markdown(note))
except ImportError:
    print(note)
print("실험 기록 저장:", note_path)

complete_lab_step(5, 5, '`lesson08_research_note.md` 파일과 기록 내용이 표시되어야 합니다.')
